# CFD-Drag-2D: U-Net flow-field surrogate (M8, v2)

Clones the repository at a fixed commit, runs the field-surrogate tests, trains the U-Net
(mask + Re -> time-averaged ux, uy, Cp) on the M7 field dataset (output of the notebook
`cfd-drag-2d-m7-fields`), and evaluates it against the mean-field and nearest-neighbour baselines
on the test split (same split as the M5 CNN).
Output (`/kaggle/working/m8_run`, plus `m8_run_seed<k>` for extra seeds): `model.pt`, `history.json`,
`metrics.json`, `errors.csv`, `fields.png`.

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
COMMIT = "main"   # branch or commit SHA
EPOCHS = 100
SEEDS = [0]       # training seeds (weight init, batch order); the split is fixed
RUN = "/kaggle/working/m8_run"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q $REPO /tmp/repo && git -C /tmp/repo checkout -q $COMMIT && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import glob, os, subprocess, torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
DATA = os.path.dirname(sorted(glob.glob("/kaggle/input/**/fields.npy", recursive=True))[0])
print("data:", DATA)
r = subprocess.run(["python", "-m", "pytest", "-q", "tests/test_field_surrogate.py"], capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, "tests failed"

In [ ]:
for seed in SEEDS:
    out = RUN if seed == 0 else f"{RUN}_seed{seed}"
    !python scripts/train_fields.py --data $DATA --out $out --epochs $EPOCHS --device cuda --seed $seed

In [ ]:
for seed in SEEDS:
    out = RUN if seed == 0 else f"{RUN}_seed{seed}"
    !python scripts/evaluate_fields.py --data $DATA --run $out
    !git rev-parse HEAD > $out/commit.txt